<a href="https://colab.research.google.com/github/sparshap95-tech/AI-Engineer/blob/main/llm_engineering_foundations/tokenization/BPE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Byte Pair Encoding (BPE) - Full Learning Example

This script demonstrates:
1. Starting from character-level tokens
2. Counting adjacent token pairs
3. Finding the most frequent pair
4. Merging the pair
5. Repeating merge steps
6. Building a subword vocabulary
7. Tokenizing a new word
8. Converting tokens to token IDs
"""

from collections import Counter


# ---------------------------------------------------------
# 1. Training corpus
# ---------------------------------------------------------

corpus = [
    "low",
    "lower",
    "lowest"
]

print("Original Corpus:")
print(corpus)


# ---------------------------------------------------------
# 2. Convert each word into character-level tokens
# ---------------------------------------------------------

tokenized_words = [list(word) for word in corpus]

print("\nInitial Character Tokens:")

for word, tokens in zip(corpus, tokenized_words):
    print(f"{word:8} -> {tokens}")


# ---------------------------------------------------------
# 3. Function to count adjacent pairs
# ---------------------------------------------------------

def get_pair_frequencies(words):
    """
    Count how many times each adjacent token pair appears.
    """

    pair_counts = Counter()

    for word in words:

        for i in range(len(word) - 1):

            pair = (word[i], word[i + 1])

            pair_counts[pair] += 1

    return pair_counts


# ---------------------------------------------------------
# 4. Function to merge one pair
# ---------------------------------------------------------

def merge_pair(words, pair):
    """
    Merge every occurrence of the selected token pair.
    """

    first, second = pair

    new_words = []

    for word in words:

        merged_word = []

        i = 0

        while i < len(word):

            # Check whether current and next token match the pair
            if (
                i < len(word) - 1
                and word[i] == first
                and word[i + 1] == second
            ):

                merged_word.append(first + second)

                i += 2

            else:

                merged_word.append(word[i])

                i += 1

        new_words.append(merged_word)

    return new_words


# ---------------------------------------------------------
# 5. Train BPE merge rules
# ---------------------------------------------------------

number_of_merges = 5

merge_rules = []

words = tokenized_words.copy()

print("\nBPE TRAINING")
print("-" * 50)

for step in range(number_of_merges):

    pair_frequencies = get_pair_frequencies(words)

    if not pair_frequencies:
        break

    # Most frequent pair
    best_pair, frequency = pair_frequencies.most_common(1)[0]

    merge_rules.append(best_pair)

    print(f"\nStep {step + 1}")
    print("Most frequent pair:", best_pair)
    print("Frequency:", frequency)

    # Merge the pair
    words = merge_pair(words, best_pair)

    print("After merge:")

    for original, updated in zip(corpus, words):
        print(f"{original:8} -> {updated}")


# ---------------------------------------------------------
# 6. Show learned merge rules
# ---------------------------------------------------------

print("\nLEARNED MERGE RULES")
print("-" * 50)

for i, rule in enumerate(merge_rules, start=1):
    print(f"{i}. {rule[0]} + {rule[1]} -> {rule[0] + rule[1]}")


# ---------------------------------------------------------
# 7. Build final vocabulary
# ---------------------------------------------------------

vocabulary = set()

# Add original characters
for word in tokenized_words:
    vocabulary.update(word)

# Add merged tokens
for first, second in merge_rules:
    vocabulary.add(first + second)

# Sort for consistent output
vocabulary = sorted(vocabulary)

print("\nFINAL VOCABULARY")
print("-" * 50)

print(vocabulary)


# ---------------------------------------------------------
# 8. Assign token IDs
# ---------------------------------------------------------

token_to_id = {
    token: index
    for index, token in enumerate(vocabulary)
}

print("\nTOKEN -> ID")
print("-" * 50)

for token, token_id in token_to_id.items():
    print(f"{token:10} -> {token_id}")


# ---------------------------------------------------------
# 9. Function to tokenize a new word using merge rules
# ---------------------------------------------------------

def apply_bpe(word, merge_rules):
    """
    Apply learned BPE merge rules to a new word.
    """

    tokens = list(word)

    print(f"\nTokenizing word: {word}")
    print("Start:", tokens)

    for first, second in merge_rules:

        i = 0

        new_tokens = []

        while i < len(tokens):

            if (
                i < len(tokens) - 1
                and tokens[i] == first
                and tokens[i + 1] == second
            ):

                new_tokens.append(first + second)

                i += 2

            else:

                new_tokens.append(tokens[i])

                i += 1

        tokens = new_tokens

        print(
            f"After rule {first}+{second} -> {first + second}:",
            tokens
        )

    return tokens


# ---------------------------------------------------------
# 10. Tokenize example words
# ---------------------------------------------------------

test_words = [
    "low",
    "lower",
    "lowest"
]

for word in test_words:

    final_tokens = apply_bpe(
        word,
        merge_rules
    )

    print("Final Tokens:", final_tokens)

    # Convert tokens to token IDs
    token_ids = [
        token_to_id[token]
        for token in final_tokens
    ]

    print("Token IDs:", token_ids)
    print("-" * 50)


# ---------------------------------------------------------
# 11. Complete pipeline example
# ---------------------------------------------------------

example_word = "lowest"

print("\nCOMPLETE PIPELINE")
print("=" * 50)

print("Raw Text:")
print(example_word)

final_tokens = apply_bpe(
    example_word,
    merge_rules
)

print("\nBPE Tokens:")
print(final_tokens)

final_ids = [
    token_to_id[token]
    for token in final_tokens
]

print("\nToken IDs:")
print(final_ids)

print(
    "\nPipeline:"
    "\nText -> BPE Tokens -> Token IDs -> Embeddings -> Transformer"
)

Original Corpus:
['low', 'lower', 'lowest']

Initial Character Tokens:
low      -> ['l', 'o', 'w']
lower    -> ['l', 'o', 'w', 'e', 'r']
lowest   -> ['l', 'o', 'w', 'e', 's', 't']

BPE TRAINING
--------------------------------------------------

Step 1
Most frequent pair: ('l', 'o')
Frequency: 3
After merge:
low      -> ['lo', 'w']
lower    -> ['lo', 'w', 'e', 'r']
lowest   -> ['lo', 'w', 'e', 's', 't']

Step 2
Most frequent pair: ('lo', 'w')
Frequency: 3
After merge:
low      -> ['low']
lower    -> ['low', 'e', 'r']
lowest   -> ['low', 'e', 's', 't']

Step 3
Most frequent pair: ('low', 'e')
Frequency: 2
After merge:
low      -> ['low']
lower    -> ['lowe', 'r']
lowest   -> ['lowe', 's', 't']

Step 4
Most frequent pair: ('lowe', 'r')
Frequency: 1
After merge:
low      -> ['low']
lower    -> ['lower']
lowest   -> ['lowe', 's', 't']

Step 5
Most frequent pair: ('lowe', 's')
Frequency: 1
After merge:
low      -> ['low']
lower    -> ['lower']
lowest   -> ['lowes', 't']

LEARNED MERGE RULES